# Практическая работа № 1
## Анализ цифрового образовательного следа средствами Google Colab и BigQuery

**Курс:** Методы анализа больших данных
**Сквозной кейс:** раннее выявление признаков учебных затруднений по цифровому образовательному следу

---

| Поле | Значение |
|---|---|
| ФИО студента | *впишите* |
| Группа | *впишите* |
| **Номер варианта** | *впишите* |
| Датасет варианта | *впишите* |
| Дата выполнения | *впишите* |

---

### Маршрут работы

Образовательная задача → проверка качества данных (Colab) → подготовка среза 30–80 МБ →
загрузка в BigQuery Sandbox → SQL-аналитика → сопоставление Python и SQL →
простое правило (сигнал) → педагогическая интерпретация и этика.

### Три правила, которые важнее кода

1. **Минимизация данных.** В работе используются только обезличенный `id_student`, временные метки
   и категории учебного контента. Демографические и чувствительные признаки в модель **не включаются**.
2. **Сигнал, а не диагноз.** Формулировка «модель выявила слабого ученика» недопустима.
   Допустимая формулировка: «получен сигнал для педагогической проверки и предложения адресной поддержки».
3. **Экономия квоты BigQuery Sandbox.** 10 GiB хранилища выдаются пожизненно и не возвращаются
   после удаления таблиц: одна загрузка файла ≤ 90 МБ на человека. `LIMIT` объём сканирования **не уменьшает**.

> Перед началом прочитайте `README.md` — там разобраны типовые ошибки (лимит 100 МБ, 60 дней жизни таблиц,
> запрет DML в Sandbox).

## Шаг 1. Подключение библиотек

Ячейку можно выполнить без изменений.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

print("Библиотеки подключены. Версия pandas:", pd.__version__)

## Шаг 2. Загрузка данных

Возьмите файл среза, выданный преподавателем (`practice01_oulad.csv` или файл вашего варианта).

Способ 1 — загрузить с компьютера в сессионное хранилище Colab:

```python
from google.colab import files
uploaded = files.upload()
```

Способ 2 — примонтировать Google Диск, если файл лежит там:

```python
from google.colab import drive
drive.mount("/content/drive")
DATA_PATH = "/content/drive/MyDrive/practice01_oulad.csv"
```

**TODO:** укажите путь к файлу и прочитайте его в DataFrame.

In [ ]:
# TODO: укажите имя вашего файла
DATA_PATH = "practice01_oulad.csv"

df = pd.read_csv(DATA_PATH)

print("Размерность датасета (строк, столбцов):", df.shape)
df.head()

## Шаг 3. Первичная разведка данных (EDA) — 2 балла

Ответьте кодом на четыре вопроса:

1. Какие столбцы и типы данных в таблице?
2. Сколько уникальных обучающихся?
3. Есть ли пропуски и в каких столбцах?
4. Каков диапазон дней (`day`) и есть ли отрицательные значения (события до старта курса)?

**TODO:** используйте `df.info()`, `df.isna().sum()`, `df["id_student"].nunique()`, `df["day"].describe()`.

In [ ]:
# ВАШ КОД ЗДЕСЬ
# df.info()
# print("Уникальных обучающихся:", ...)
# print(df.isna().sum())
# df["day"].describe()

### Вывод по качеству данных (Veracity)

**TODO:** запишите в 2–3 предложениях, что вы увидели: есть ли пропуски, дубликаты,
подозрительные значения (например, аномально большое число кликов за один день).
Сформулируйте, как это может повлиять на выводы.

> *Ваш текст:*

## Шаг 4. Признаки первых 30 дней (Feature Engineering) — часть SQL-блока

Единица наблюдения в исходной таблице — **одно событие** (клик).
Для анализа нужно перейти на уровень **обучающегося**.

**TODO:**
1. Отфильтруйте события первых 30 дней (`day` от 0 до 29).
2. Сгруппируйте по `id_student` и рассчитайте:
   - `total_clicks` — сумму кликов,
   - `active_days` — число уникальных активных дней,
   - `resources_used` — число уникальных ресурсов,
   - `final_result` — итог курса (первое значение).

In [ ]:
# ВАШ КОД ЗДЕСЬ
# early = df[(df["day"] >= 0) & (df["day"] < 30)]
# student_metrics = early.groupby("id_student").agg(
#     total_clicks=("sum_click", "sum"),
#     active_days=("day", "nunique"),
#     resources_used=("id_site", "nunique"),
#     final_result=("final_result", "first"),
# ).reset_index()
# student_metrics.head()

## Шаг 5. Простое правило педагогической поддержки — 2 балла

Модель здесь не нужна: достаточно прозрачной эвристики, которую можно объяснить коллеге-педагогу.

**TODO:**
1. Рассчитайте 25-й перцентиль (`quantile(0.25)`) по `total_clicks` и по `active_days`.
2. Создайте булев флаг `support_signal` — студент ниже обоих порогов.
3. Сравните долю успешного завершения курса в группе с сигналом и без него.

> Успешным считается `final_result` из набора `["Pass", "Distinction"]`.

In [ ]:
# ВАШ КОД ЗДЕСЬ
# q25_clicks = ...
# q25_days = ...
# student_metrics["support_signal"] = ...
# student_metrics["is_success"] = ...
# student_metrics.groupby("support_signal")["is_success"].agg(["count", "mean"])

## Шаг 6. Визуализация — входит в оценку EDA

**TODO:** постройте **один** график, который делает вашу мысль наглядной. Варианты:
гистограмма распределения `total_clicks`, столбчатая диаграмма доли успешных по группам,
график накопленной активности для нескольких студентов.

Не забудьте подписи осей и заголовок на русском языке.

In [ ]:
# ВАШ КОД ЗДЕСЬ
# plt.figure(figsize=(8, 5))
# ...
# plt.title("...")
# plt.xlabel("...")
# plt.ylabel("...")
# plt.tight_layout()
# plt.show()

## Шаг 7. Выгрузка файла для BigQuery

**Проверьте размер перед загрузкой.** Веб-консоль BigQuery принимает локальный файл
строго **до 100 МБ**, а рекомендуемый учебный срез — **30–80 МБ**.

Ячейка ниже сохраняет файл и печатает его размер.

In [ ]:
UPLOAD_NAME = "my_upload_table.csv"

# TODO: при необходимости оставьте только нужные столбцы, чтобы уменьшить размер
# df_to_upload = df[["id_student", "day", "id_site", "sum_click", "activity_type", "final_result"]]
df_to_upload = df

df_to_upload.to_csv(UPLOAD_NAME, index=False)
size_mb = Path(UPLOAD_NAME).stat().st_size / 1024**2
print(f"Файл сохранён: {UPLOAD_NAME} | строк: {len(df_to_upload):,} | размер: {size_mb:.2f} МБ")

if size_mb > 90:
    print("ВНИМАНИЕ: файл слишком большой. Сократите период, курс или набор столбцов.")
else:
    print("Размер в норме — можно загружать в BigQuery Sandbox.")

# Скачать файл на компьютер:
# from google.colab import files
# files.download(UPLOAD_NAME)

## Шаг 8. SQL-аналитика в BigQuery Sandbox — 4 балла

Порядок действий подробно описан в `README.md`, раздел «Этап Б». Кратко:

1. console.cloud.google.com/bigquery → **New Project** (`edu-analytics-sandbox`).
2. Explorer → ⋮ у проекта → **Create dataset** → ID `learning_analytics`, локация `EU` или `US`.
3. ⋮ у датасета → **Create table** → Create table from: **Upload** → ваш CSV →
   Table: `oulad_events` → Schema: **Auto detect** → Advanced options → **Header rows to skip: 1**.
4. Проверьте вкладку **Schema**: числовые поля должны быть `INTEGER`, а не `STRING`.
5. **Query → In new tab** и выполните запросы.

> В Sandbox запрещены `INSERT`, `UPDATE`, `DELETE`, `MERGE` и потоковая вставка.
> Вся аналитика — только `SELECT`, CTE (`WITH`) и оконные функции.

### Запрос 1 — агрегация (`GROUP BY`)

**TODO:** вставьте сюда текст вашего запроса и 3–5 строк результата.

```sql
-- ВАШ ЗАПРОС
```

**Результат:**

| ... | ... |
|---|---|

### Запрос 2 — обязательная оконная функция вашего варианта

**TODO:** вставьте запрос с оконной функцией, указанной в таблице вариантов
(`SUM() OVER`, `LAG()`, `PERCENT_RANK()`, `ROW_NUMBER()`, `NTILE()` и т. п.), и результат.

```sql
-- ВАШ ЗАПРОС
```

**Результат:**

| ... | ... |
|---|---|

### Сверка Python и SQL — 2 балла (часть SQL-блока)

**TODO:** совпали ли значения `total_clicks` для 2–3 конкретных студентов в pandas и в BigQuery?
Если нет — найдите причину (разный фильтр по дням, пропуски, тип столбца).

> *Ваш комментарий:*

In [ ]:
# Необязательная проверка: вставьте выгруженный из BigQuery результат Запроса 1 (CSV) и сравните
# bq_df = pd.read_csv("bq_query1_result.csv")
# merged = student_metrics.merge(bq_df, on="id_student", suffixes=("_py", "_sql"))
# print((merged["total_clicks_py"] != merged["total_clicks_sql"]).sum(), "расхождений")

## Шаг 9. Педагогическая интерпретация и этический вывод — 2 балла

Напишите связный текст на **5–7 предложений**, отвечая на три вопроса:

1. Что на самом деле показывает низкая ранняя активность обучающегося?
   Назовите минимум три альтернативных объяснения, не связанных со способностями.
2. Почему такого студента нельзя автоматически считать «неуспевающим»?
   Сошлитесь на принцип минимизации данных (152-ФЗ, приказ РКН № 140 от 2025 г., GDPR).
3. Какое конкретное педагогическое действие будет этически корректным:
   что, кому, на какой день и в какой формулировке вы сообщите?

> *Ваш вывод:*

---

### Чек-лист самопроверки перед сдачей

- [ ] Заполнена шапка: ФИО, группа, номер варианта.
- [ ] Выполнен EDA: типы, пропуски, число уникальных студентов, диапазон дней.
- [ ] Есть агрегация по студентам и корректный переход с уровня событий на уровень человека.
- [ ] Реализовано прозрачное правило (сигнал) с явно названными порогами.
- [ ] Построен минимум один подписанный график.
- [ ] Файл для загрузки ≤ 90 МБ, загружен в BigQuery **один раз**.
- [ ] Приведены два SQL-запроса, второй — с оконной функцией вашего варианта.
- [ ] Выполнена сверка результатов Python и SQL.
- [ ] Написан педагогический вывод без формулировок-ярлыков.
- [ ] В работе нет ФИО, контактов и демографических признаков обучающихся.